# Module 19 project-exercise demo — Paired comparison, critical slices, and verified actions

**Implementation demo, not a completed full-scale project.** Two deterministic rules process 200 fictional refund requests. Responses are written to JSON, reloaded, and used for paired bootstrap analysis without further calls. A second experiment compares ordinary requests, critical exceptions, and actual local ledger changes. A final probe really searches documents with a deliberately limited reader; it validates the protocol rather than measuring an LLM.
All inputs are small, deterministic, offline, and CPU-friendly. The goal is to
validate interfaces, metrics, invariants, and report structure before spending
GPU or API budget.

In [ ]:
from __future__ import annotations
import importlib.util, subprocess, sys
for package in ("numpy", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


import math
import random
import time
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np

SEED = 20260801
random.seed(SEED)
np.random.seed(SEED)
print(f"reproducible smoke tier; seed={SEED}")
DEMO_LANGUAGE = 'en'


Locate the days=30 boundary before running the cell. A accepts that day but ignores packaging; B checks packaging yet excludes the final allowed day. Every ordinary request has intact packaging. Once responses are saved, all statistics use reloaded JSON rather than calling either rule again.

In [ ]:
import math
from dataclasses import dataclass
from typing import Callable, Mapping, Sequence


@dataclass(frozen=True)
class Example:
    example_id: str
    prompt: str
    expected: str
    slice_name: str

    def __post_init__(self):
        if any(not isinstance(v, str) or not v.strip()
               for v in (self.example_id, self.prompt, self.expected, self.slice_name)):
            raise ValueError("example fields must be non-empty strings")


def exact_match(prediction: str, expected: str) -> float:
    return float(prediction.strip().casefold() == expected.strip().casefold())


def evaluate(
    model_fn: Callable[[str], str],
    dataset: Sequence[Example],
    metrics: Mapping[str, Callable[[str, str], float]],
) -> dict:
    dataset = tuple(dataset)
    if not callable(model_fn) or not metrics or any(not isinstance(k, str) or not k or not callable(v) for k, v in metrics.items()):
        raise ValueError("supply a callable model and named callable metrics")
    if not dataset:
        raise ValueError("dataset must be non-empty")
    if any(not isinstance(ex, Example) for ex in dataset):
        raise TypeError("expected Example records")
    if len({ex.example_id for ex in dataset}) != len(dataset):
        raise ValueError("example_id values must be unique")

    records = []
    for ex in dataset:
        prediction = model_fn(ex.prompt)  # the only model call for this item
        if not isinstance(prediction, str):
            raise TypeError("model response must be text")
        scores = {
            name: metric(prediction, ex.expected)
            for name, metric in metrics.items()
        }
        if any(isinstance(v, bool) or not isinstance(v, (float, int)) or not math.isfinite(v) or not 0 <= v <= 1 for v in scores.values()):
            raise ValueError("metric scores must be finite numbers in [0, 1]")
        records.append({
            "id": ex.example_id,
            "slice": ex.slice_name,
            "expected": ex.expected,
            "prediction": prediction,
            "scores": scores,
        })

    aggregate = {
        name: sum(row["scores"][name] for row in records) / len(records)
        for name in metrics
    }
    return {
        "records": records,
        "aggregate": aggregate,
        "model_calls": len(records),
    }

def _positive_int(name: str, value: int) -> int:
    """Validate and return a strictly positive, non-boolean integer."""
    if isinstance(value, bool) or not isinstance(value, int) or value <= 0:
        raise ValueError(f"{name} must be a positive integer")
    return value

def _non_negative_int(name: str, value: int) -> int:
    """Validate and return a non-negative, non-boolean integer."""
    if isinstance(value, bool) or not isinstance(value, int) or value < 0:
        raise ValueError(f"{name} must be a non-negative integer")
    return value

def _positive_finite(name: str, value: float) -> float:
    """Validate and return a finite positive real number."""
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise ValueError(f"{name} must be a real number")
    number = float(value)
    if not math.isfinite(number) or number <= 0.0:
        raise ValueError(f"{name} must be finite and positive")
    return number

def wilson_interval(successes: int, total: int, z_value: float = 1.96) -> tuple[float, float]:
    """Return the Wilson score interval for a binomial proportion."""
    n = _positive_int("total", total)
    s = _non_negative_int("successes", successes)
    if s > n:
        raise ValueError("successes cannot exceed total")
    z_value = _positive_finite("z_value", z_value)
    p = s / n
    z2 = z_value**2
    denominator = 1.0 + z2 / n
    centre = (p + z2 / (2.0 * n)) / denominator
    radius = z_value * math.sqrt(p * (1.0 - p) / n + z2 / (4.0 * n**2)) / denominator
    return max(0.0, centre - radius), min(1.0, centre + radius)

def mcnemar_exact_pvalue(first_only: int, second_only: int) -> float:
    """Return the two-sided exact McNemar p-value for discordant paired outcomes."""
    first_only = _non_negative_int("first_only", first_only)
    second_only = _non_negative_int("second_only", second_only)
    discordant = first_only + second_only
    if discordant == 0:
        return 1.0
    tail = min(first_only, second_only)
    cdf = sum(math.comb(discordant, k) for k in range(tail + 1)) / (1 << discordant)
    return min(1.0, 2.0 * cdf)
import json, tempfile
from pathlib import Path
# Authored fictional policy: sealed items may be returned through day 30 inclusive.
# Unique requests, but generated fixtures are NOT a sample of real customers.
dataset = tuple(Example(
    f"ordinary-{i:03d}", json.dumps({"days": i % 50, "sealed": True}),
    "refund" if i % 50 <= 30 else "deny", "ordinary"
) for i in range(200))
def policy_a(prompt):
    item = json.loads(prompt)
    return "refund" if item["days"] <= 30 else "deny"  # ignores sealed flag
def policy_b(prompt):
    item = json.loads(prompt)
    return "refund" if item["sealed"] and item["days"] < 30 else "deny"  # boundary bug

reports = {name: evaluate(fn, dataset, {"exact": exact_match})
           for name, fn in [("A", policy_a), ("B", policy_b)]}
passport = {"fixture": "fictional-refunds-v1", "systems": ["A-v1", "B-v1"],
            "trial": 0, "metric": "casefold-exact-v1", "seed": 1901}
payload = {"passport": passport, "reports": reports}
with tempfile.TemporaryDirectory(dir=Path.cwd()) as folder:
    file = Path(folder)/"saved_responses.json"
    file.write_text(json.dumps(payload), encoding="utf-8")
    saved = json.loads(file.read_text(encoding="utf-8"))
# All rescoring below uses saved responses, making no new policy calls.
for name in ("A", "B"):
    rows = saved["reports"][name]["records"]
    assert all(exact_match(r["prediction"], r["expected"]) == r["scores"]["exact"] for r in rows)
a = np.array([r["scores"]["exact"] for r in saved["reports"]["A"]["records"]])
b = np.array([r["scores"]["exact"] for r in saved["reports"]["B"]["records"]])
assert len(a) == 200 and a.sum() == 200 and b.sum() == 196
rng = np.random.default_rng(1901)
differences = a - b
bootstrap = differences[rng.integers(0, len(a), size=(5000, len(a)))].mean(axis=1)
interval = np.quantile(bootstrap, [.025, .975])
first_only, second_only = int(((a==1)&(b==0)).sum()), int(((a==0)&(b==1)).sum())
p_value = mcnemar_exact_pvalue(first_only, second_only)
assert (first_only, second_only) == (4, 0) and p_value == .125
print("A/B:", a.mean(), b.mean(), "paired delta:", differences.mean(),
      "bootstrap:", interval, "exact McNemar:", p_value)
print("Calls:", {n: r["model_calls"] for n, r in reports.items()},
      "latency/cost: unmeasured; deterministic rules, not LLMs")
assert np.isclose(interval[0], .005) and np.isclose(interval[1], .04)

A scores 200/200 and B 196/200. All four disagreements favor A, giving exact McNemar p=0.125. The empirical bootstrap interval nevertheless excludes zero. This small boundary sample exposes a limitation of that interval; it does not establish a real-world win for A.

Change the property under test. Critical cases require rejection of unsealed returns; action checks reject unauthorized and duplicate effects. The final document experiment varies where the target line falls relative to the reader's boundaries. Its outcome follows from the visible search implementation.

In [ ]:
# Predeclared independent gates: ordinary >= .99, critical = 1, trajectory = 1.
# They illustrate a decision rule; they are not a justified production risk policy.
critical = tuple(Example(
    f"critical-{i}", json.dumps({"days": i, "sealed": False}), "deny", "critical"
) for i in range(20))
quality = {name: {
    "ordinary": reports[name]["aggregate"]["exact"],
    "critical": evaluate(fn, critical, {"exact":exact_match})["aggregate"]["exact"]
} for name, fn in [("A",policy_a),("B",policy_b)]}

def run_trajectory(authorized, version):
    """Execute two delivery attempts; inspect effects rather than final words."""
    ledger, seen, events = [], set(), []
    for _ in range(2):
        operation_id = "refund-17"
        if version == "B":
            if not authorized:
                events.append("denied")
                continue
            if operation_id in seen:
                events.append("replay")
                continue
        ledger.append({"operation":operation_id, "amount":7})
        seen.add(operation_id)
        events.append("effect")
    expected = [{"operation":"refund-17", "amount":7}] if authorized else []
    return {"ok":ledger == expected, "ledger":ledger, "events":events}

traces = {name: [run_trajectory(allowed,name) for allowed in [True,False]]
          for name in ("A","B")}
for name in ("A","B"):
    quality[name]["trajectory"] = sum(t["ok"] for t in traces[name])/2
gates = {"ordinary":.99, "critical":1., "trajectory":1.}
decision = {name: all(scores[k] >= gates[k] for k in gates) for name,scores in quality.items()}
assert quality == {"A":{"ordinary":1., "critical":0., "trajectory":0.},
                   "B":{"ordinary":.98, "critical":1., "trajectory":1.}}
assert decision == {"A":False,"B":False}
print("suites:", quality, "release gates:", decision)
print("actual traces:", traces)
# There is no justified universal average over these deliberately different suites.
# Fix B's boundary and rerun on development data; keep a fresh final audit set.

# A public reproducible canary verifies the detector, not privacy or training data.
import hashlib
canary = hashlib.sha256(b"public-m19-positive-control").hexdigest()[:32]
assert canary not in "ordinary answer"
assert canary in ("planted positive control " + canary)
print("canary detector: negative and planted positive controls pass")

# Actual search, with a deliberately limited reader as the negative control.
# Length counts lines, NOT model tokens. No model is involved.
positions = np.linspace(0, 1, 9)
measured = {}
for length in (128, 512, 1024):
    full, edges = [], []
    for fraction in positions:
        pos = round(fraction * (length-1))
        lines = [f"irrelevant line {j}" for j in range(length)]
        needle = "TARGET: blue-orchid"
        lines[pos] = needle
        extract = lambda selected: next((s.split(": ",1)[1] for s in selected if s.startswith("TARGET: ")), "")
        full.append(extract(lines) == "blue-orchid")
        edges.append(extract(lines[:32]+lines[-32:]) == "blue-orchid")
    assert all(full)
    measured[length] = edges
    plt.plot(positions, edges, "o-", label=f"{length} "+("строк" if DEMO_LANGUAGE=="ru" else "lines"))
plt.plot(positions, full, "--", color="black", label="full scan")
plt.xlabel("позиция иголки" if DEMO_LANGUAGE=="ru" else "needle position")
plt.ylabel("успех извлечения (0/1)" if DEMO_LANGUAGE=="ru" else "retrieval success (0/1)")
plt.title("Проверка ограниченного чтения" if DEMO_LANGUAGE=="ru" else "Testing a limited reader")
plt.legend(); plt.tight_layout(); plt.show()
assert measured[128] == [True,True,False,False,False,False,False,True,True]
assert measured[512] == measured[1024] == [True,False,False,False,False,False,False,False,True]

The critical slice reverses the ranking, and neither version satisfies every criterion. State checks reveal failures a final message would miss. Full scanning retrieves the target everywhere; the restricted reader succeeds only near the edges. These are measurements of two search programs, not of an LLM's context use.

## Scaling to the full project

Replace the rules with versioned systems and collect independent, appropriately sourced tasks. Bootstrap sessions when multiple rows share one session. Fixture scores and intervals demonstrate the procedure; repeated templates do not establish generalization to customers. Here the percentile bootstrap excludes zero while exact McNemar gives p=0.125. With few discrete disagreements at a boundary, the empirical bootstrap cannot generate a B-only success that was never observed. Treat this as a limitation of the interval, not a proven win. Record measured latency and cost, marking unavailable values as missing. After fixing day=30, use a fresh final audit split: repeatedly inspected regression cases become development evidence. The local ledger check is single-process and does not model the network failures or durable transactions studied in M18.

### Minimum report

1. Data, model, and code revisions, plus the random seed.
2. The primary metric, plus quality, cost, latency, and memory where applicable.
3. A baseline, a matched budget, and at least one negative control.
4. Failed runs and the limits of the supported conclusions.